<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/SegmentationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub
import kagglehub, pathlib, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
print("downloaded to:", path)

100%|██████████| 21.9G/21.9G [04:33<00:00, 85.8MB/s]

Extracting files...


downloaded to: /root/.cache/kagglehub/datasets/yaroslavchyrko/rescuenet/versions/1


In [ ]:
root = pathlib.Path(path)
org_dir   = list(root.rglob("train-org-img"))[0]
label_dir = list(root.rglob("train-label-img"))[0]
print("images:", len(list(org_dir.glob("*.jpg"))), "| masks:", len(list(label_dir.glob("*.png"))))

images: 3595 | masks: 3595


In [ ]:
#0=Unlabeled, 1= Water, 2 = Building w/o Damage
#3 Building with minor damage, 4 Building with Major damage
#5 Building completely destroyed, 6 Vechicle, 7 Clear Road
#8 Blocked Road, 9 Tree, 10 Pool

!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp
import torch, torch.nn as nn, torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
class_number = 11

model = smp.Unet(encoder_name="resnet34",
                 encoder_weights="imagenet",
                 in_channels=3,
                 classes=class_number,
                 )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")
root = pathlib.Path(path)

train_orginal   = list(root.rglob("train-org-img"))[0]
train_label = list(root.rglob("train-label-img"))[0]
val_orginal     = list(root.rglob("val-org-img"))[0]
val_label   = list(root.rglob("val-label-img"))[0]
print("train imgs/masks:", len(list(train_orginal.glob('*.jpg'))), len(list(train_label.glob('*.png'))))
print("val imgs/masks:  ", len(list(val_orginal.glob('*.jpg'))), len(list(val_label.glob('*.png'))))

Using Colab cache for faster access to the 'rescuenet' dataset.
train imgs/masks: 3595 3595
val imgs/masks:   449 449


In [ ]:
loss = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass") # diceloss is used common in image segmentation by focusing on the intersection of predicted mask and the real mask

def criterion(logits, masks):
  return loss(logits, masks) + dice_loss(logits, masks)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, 0.9)

In [ ]:
import pathlib, numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

In [ ]:
#RescueNetSegmentedDataset class, Kento

In [ ]:
#build the data sets from org_dir, label_dir, see ResNetModel for reference, Oluj
#equivalent to class LadiDataset(Dataset):
class RescueNetDataset(Dataset):
    def init(self, org_dir, label_dir, size = 512, train =True):
        self.org_dir = pathlib.Path(org_dir)
        self.label_dir = pathlib.Path(label_dir)
        self.images = sorted(self.org_dir.glob("*.jpg"))
        self.size = size
        self.train = train

    def len(self):
        return len(self.images)

    def getitem(self, idx):
        img_path = self.images[idx]
        mask_path = self.label_dir / f"{img_path.stem}_lab.png"

        image =  Image.open(img_path).convert("RGB") #converted to RGB channels because it's n image
        mask = Image.open(mask_path) #leaving as single channel image with integer ids.

        image = image.resize((self.size,self.size), Image.BILINEAR) #resizing to chosen size, used BILINEAR for interpolation
        mask = mask.resize((self.size,self.size), Image.NEAREST) #masks needs no averaging, so everythings stays as an integer

        if self.train and torch.rand(1).item() < 0.5:
            image = TF. hflip(image); mask = TF.hflip(mask) #keeps any transformation that's applied to the image applied to the mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) #this line and the line before converts the image into the numeric form the pretrained model was trained on,
        #they convert a bunch of stuff like from PIL image to a PyTorch tensor, rescales pixel values down to 0-1 floats, etc.
        mask = torch.as_tensor(np.array(mask), dtype = torch.long) #similar thing converts the mask to a long tensor shape
        return{"image": image, "mask": mask} #returns the image and mask as Pytorch tensors, image is a float32, while mask is a long(no color channel dimension)

In [ ]:
#IoU or mIOU evaluation function, Aishani
def calc_IOU(pred_arr, label_arr):
  #multiclass - each class has separate IOU score?
  iou_scores = []
  #union pix
  for c_index in range(class_number):
    #get where they overlap and are correct
    class_gt = label_arr == c_index
    class_pred_area = pred_arr == c_index
    intersect_pix = np.sum((class_gt & class_pred_area))
    intersect_pix = float(intersect_pix)

    #get total area of masks merged

    total_region = np.sum((class_gt | class_pred_area))
    total_region = float(total_region)

    if(total_region == 0):
      continue

    iou_scores.append(intersect_pix/total_region)

  return iou_scores

def calc_mean_IOU(iou_scores): # can merge into another output above
  #iou_scores = [score for score in iou_scores if score != 0]
  return np.mean(iou_scores)


#testing
# import PIL
# import os
# from pathlib import Path
# import matplotlib.pyplot as plt
# c = 0
# p_img_masks = []
# for f in Path(train_label).iterdir():
#   if c<5:
#     img = PIL.Image.open(f)
#     img_arr = np.array(img)
#     p_img_masks.append(img_arr)
#   c+=1

# print(calc_IOU(p_img_masks[4],p_img_masks[1]))
# fix ,ax = plt.subplots(1,2,figsize=(6,6))

# ax[0].imshow(p_img_masks[4])
# ax[1].imshow(p_img_masks[1])

In [ ]:
import torch
import torchvision
from dataset import RescueNetDataset
from torch.utils.data import DataLoader

def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])

def get_loaders(train_dir, train_maskdir, val_dir, val_maskdir, batch_size, train_transform, val_transform, num_workers=4, pin_memory=True):
    train_dataset = RescueNetDataset(image_dir=train_dir, mask_dir = train_maskdir, transform=train_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, shuffle=True)

    val_dataset = RescueNetDataset(image_dir = val_dir, mask_dir = val_maskdir, transform=val_transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory)

    return train_loader, val_loader

def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_pixels = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            preds = torch.argmax(model(x), dim=1)
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)

    print(f"Got {num_correct}/{num_pixels} with accuracy {num_correct/num_pixels*100:.2f}")
    model.train()

In [ ]:
# Class for actual U-Net architecture
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

class DoubleConv(nn.Module): # two convolutions: conv2d 3x3 then ReLU two times
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNET(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNET, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # down part of unet
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature)) # register new layer as submodule
            in_channels = feature # make num of input channels into the last iterations number of output channels

        # up part
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)) # Tranposed Convolution for upsampling: feature*2 because of concatenated matrices
            self.ups.append(DoubleConv(feature*2, feature)) # Double conv for each layer of upsampling.
            # each up layer: transpose conv, then double conv

        # bottom bottelneck layer
        self.bottleneck = DoubleConv(features[-1], features[-1]*2) # double conv: in is 512, out is 1024 to increase feature channels to its max

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1) # final layer: done! single convolution with in 64, out 1

    def forward(self, x):
        skip_connections = []

        for down in self.downs: # "for every layer in encoder:"
            x = down(x)         # run x through the layer
            skip_connections.append(x) # then save it to skip_connections to be catted later
            x = self.pool(x)    # pool it (downsample) uses max pooling to shrink in half

        x = self.bottleneck(x)  # once encoding is done, run through bottleneck layer
        skip_connections = skip_connections[::-1] # reverse skip_connections for convenience so we can use it in decoding

        for idx in range(0, len(self.ups), 2): # for each layer in decoder. 8 layers, step 2 at a time since theres 2 steps at each of 4 layers
            x = self.ups[idx](x)# run x through the first step of the up layer
            skip_connection = skip_connections[idx//2] # make local var skip_connection from skip_connections list we made in encoding

            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:]) # resize so catting works. [2:] gets height and width of tensor

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip) # run x with skip connection this time through the second step of the layer

        return self.final_conv(x) # do the final layer single conv and return the result. done!


def test(): # test to make sure the size of what you input into the unet comes back the same size
    x = torch.randn((3,1,160,160))
    model = UNET(in_channels=1, out_channels=1)
    preds = model(x)
    print(preds.shape)
    print(x.shape)
    assert preds.shape == x.shape

if __name__ == "__main__":
    test()

In [ ]:
#training, saving checkpoints to drive, Brian
import torch
from torchvision.transforms import v2
from torchvision import tv_tensors
from tqdm import tqdm
import os
import torch.nn as nn
import torch.optim as optim
from model import UNET
from utils import get_loaders, save_checkpoint, load_checkpoint, check_accuracy
import kagglehub

path = kagglehub.dataset_download("yaroslavchyrko/rescuenet")

#hyperparameters

alpha = 1e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
num_epochs = 3
num_workers = 2
image_height =160
image_width = 240
pin_memory = True
load_model = True
train_img_dir = os.path.join(path, "RescueNet", "train", "train-org-img")
train_mask_dir = os.path.join(path, "RescueNet", "train", "train-label-img")
val_img_dir = os.path.join(path, "RescueNet", "val", "val-org-img")
val_mask_dir = os.path.join(path, "RescueNet", "val", "val-label-img")

def train(loader, model, optimizer, loss_func, scaler):
    loop = tqdm(loader)

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=device)
        targets = targets.to(device=device)

        with torch.amp.autocast(device_type=device):
            predictions = model(data)
            loss = loss_func(predictions, targets)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loop.set_postfix(loss=loss.item())

def main():
    train_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.RandomRotation(degrees=35),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.1),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
    ])

    val_transforms = v2.Compose([
        v2.Resize((image_height, image_width)),
        v2.ToImage(),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64, "others": None}, scale=True),
    ])

    model = UNET(in_channels=3, out_channels=3).to(device)
    loss_func = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=alpha)

    train_loader, val_loader = get_loaders(train_img_dir, train_mask_dir,
                                           val_img_dir, val_mask_dir,
                                           batch_size,
                                           train_transforms, val_transforms,
                                           num_workers, pin_memory)

    scaler = torch.amp.GradScaler()

    for epoch in range(num_epochs):
        train(train_loader, model, optimizer, loss_func, scaler)

        checkpoint = {"state_dict": model.state_dict(),
                      "optimizer": optimizer.state_dict()}
        save_checkpoint(checkpoint)

        check_accuracy(val_loader,model,device=device)

if __name__ == "__main__":
    main()

In [ ]:
#run it